# W&B Automations tutorial (API)

This notebook walks through managing W&B Automations with the Python API: list, get, create, update, and delete. It creates two example automations:

1. **Project automation**: Run failure alert (Slack notification when a run fails).
2. **Registry automation**: Alias added to webhook (when `production` alias is added to an artifact in a collection, trigger a webhook).

**Prerequisites:** A W&B project, a Slack integration and a webhook configured in Team Settings, and (for the registry example) an artifact collection. Replace `my-team`, `my-project`, and `my-model` with your entity, project, and collection names.

## List automations

In [ ]:
import wandb

api = wandb.Api()

for automation in api.automations(entity="my-team"):
    print(automation.name, "|", getattr(automation, "scope", ""), "|", getattr(automation, "enabled", True))

## Get one automation

In [ ]:
automation = api.automation(name="run-failure-alert", entity="my-team")
print(automation.name, automation.description, automation.enabled)

## Create: Project automation (run failure to Slack)

In [ ]:
from wandb.automations import OnRunState, RunEvent, SendNotification

project = api.project("my-project", entity="my-team")
slack_integration = next(api.slack_integrations(entity="my-team"))

event = OnRunState(
    scope=project,
    filter=RunEvent.state.in_(["failed"]),
)
action = SendNotification.from_integration(slack_integration)

automation = api.create_automation(
    event >> action,
    name="run-failure-alert",
    description="Notify the team when a run fails.",
)
print("Created:", automation.name)

## Create: Registry automation (alias added to webhook)

In [ ]:
from wandb.automations import OnAddArtifactAlias, ArtifactEvent, SendWebhook

collection = api.artifact_collection(name="my-model", type_name="model")
webhook_integration = next(api.webhook_integrations(entity="my-team"))

event = OnAddArtifactAlias(
    scope=collection,
    filter=ArtifactEvent.alias.eq("production"),
)
action = SendWebhook.from_integration(
    webhook_integration,
    payload={
        "event": "${event_type}",
        "model": "${artifact_collection_name}",
        "version": "${artifact_version_string}",
    },
)

automation = api.create_automation(
    event >> action,
    name="production-alias-webhook",
    description="Trigger webhook when production alias is added.",
)
print("Created:", automation.name)

## Update an automation

In [ ]:
automation = api.automation(name="run-failure-alert", entity="my-team")
updated = api.update_automation(
    automation,
    enabled=False,
    description="Temporarily disabled.",
)
print("Updated:", updated.name, updated.enabled, updated.description)

## Delete an automation

In [ ]:
# Uncomment to delete. Replace with your automation name.
# api.delete_automation(api.automation(name="run-failure-alert", entity="my-team"))
# Or by ID: api.delete_automation(automation.id)